First we import the necessary modules to run the notebook, and we connect the Jupyter Notebook to the Qdrant Client

In [ ]:
import pandas as pd
import numpy as np
import joblib

from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, Range, MatchValue
from sentence_transformers import SentenceTransformer

# Setup Connections and Models

client = QdrantClient(host="localhost", port=6333)

model = SentenceTransformer("all-MiniLM-L6-v2")

scaler = joblib.load("../artifacts/metadata_scaler.pkl")

COLLECTION_NAME = "book_recommender"


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5372.36it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Query Vector Builder

def build_query_vector(
    text,
    rating=4.0,
    word_count=300,
    tag_count=10
):
    """
    Builds 387-dim hybrid semantic+metadata vector
    """

    semantic_vector = model.encode(text)

    metadata_vector = np.array([
        [word_count, tag_count, rating]
    ])

    scaled_meta = scaler.transform(metadata_vector).flatten()

    combined_vector = np.hstack([
        semantic_vector,
        scaled_meta
    ])

    return combined_vector.tolist()



In [ ]:
# Hybrid Semantic Search

def search_books(
    query_text,
    mood=None,
    min_rating=0.0,
    limit=5
):

    vector = build_query_vector(query_text)

    must_conditions = []

    if mood:

        must_conditions.append(
            FieldCondition(
                key="moods",
                match=MatchValue(value=mood)
            )
        )

    if min_rating > 0:

        must_conditions.append(
            FieldCondition(
                key="average_rating",
                range=Range(gte=min_rating)
            )
        )

    query_filter = Filter(must=must_conditions) if must_conditions else None


    results = client.search(
        collection_name=COLLECTION_NAME,
        query_vector=vector,
        query_filter=query_filter,
        limit=limit
    )


    recommendations = []

    for hit in results:

        row = hit.payload.copy()

        row["similarity_score"] = hit.score

        recommendations.append(row)


    return (
        pd.DataFrame(recommendations)
        .sort_values("similarity_score", ascending=False)
        .reset_index(drop=True)
    )


# =====================================================
# Example Searches
# =====================================================

print("Search: Dark mystery, rating > 4.0")

display(
    search_books(
        "A dark mystery in a small town",
        min_rating=4.0
    )
)


print("\nSearch: Space adventure, escapist mood")

display(
    search_books(
        "Space adventure",
        mood="escapist"
    )
)